# Customer Review Decision Engine (Strict Version)

This notebook builds a review intelligence system for rigorous academic and business reporting:
- Reproducible random sampling from MongoDB
- Explicit data quality checks
- Leakage-safe train/validation/test workflow
- Model selection via cross-validation on train only
- Final evaluation on untouched test set
- Actionable product decisions from review intelligence

## 1) Imports and Global Config

All random seeds are fixed for reproducibility.

In [101]:
import re
import json
import random
import numpy as np
import pandas as pd

from pymongo import MongoClient
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

## 2) Load Data from MongoDB (Full Dataset by Default)

Strict protocol:
- Default mode uses **all documents** in the selected collection.
- Optional random sampling mode is preserved for quick pilot experiments.
- Stable schema is enforced at ingestion time.

In [102]:
MONGO_URI = "mongodb://localhost:27017/"
USE_FULL_DATA = True        # Strict mode requested by user/professor
SAMPLE_SIZE = 100000        # Used only when USE_FULL_DATA = False

# Set explicit database/collection for reproducible runs across machines.
PREFERRED_DB = "bigdata"
PREFERRED_COLLECTION = "review"

client = MongoClient(MONGO_URI)

skip_dbs = {"admin", "config", "local"}
selected_db, selected_col = None, None

if PREFERRED_DB and PREFERRED_COLLECTION:
    if PREFERRED_DB in client.list_database_names() and PREFERRED_COLLECTION in client[PREFERRED_DB].list_collection_names():
        selected_db, selected_col = PREFERRED_DB, PREFERRED_COLLECTION
    else:
        print("Preferred DB/collection not found; falling back to auto-detection.")

if selected_db is None:
    candidates = []
    for db_name in client.list_database_names():
        if db_name in skip_dbs:
            continue
        db_tmp = client[db_name]
        for col_name in db_tmp.list_collection_names():
            cnt = db_tmp[col_name].count_documents({})
            if cnt > 0:
                candidates.append((cnt, db_name, col_name))

    if not candidates:
        raise RuntimeError("No non-system MongoDB collection with data found.")

    candidates.sort(reverse=True)
    _, selected_db, selected_col = candidates[0]

collection = client[selected_db][selected_col]
doc_count = collection.count_documents({})

print(f"Using database: {selected_db}")
print(f"Using collection: {selected_col}")
print(f"Total documents: {doc_count:,}")

project_stage = {
    "$project": {
        "_id": 0,
        "Id": 1,
        "ProductId": 1,
        "UserId": 1,
        "Score": 1,
        "Summary": 1,
        "Text": 1,
        "ProductURL": 1,
        "HelpfulnessNumerator": 1,
        "HelpfulnessDenominator": 1,
        "Time": 1
    }
}

if USE_FULL_DATA:
    pipeline = [project_stage]
    run_label = "FULL"
else:
    pipeline = [
        {"$sample": {"size": min(SAMPLE_SIZE, doc_count)}},
        project_stage
    ]
    run_label = f"SAMPLE_{min(SAMPLE_SIZE, doc_count)}"

rows = list(collection.aggregate(pipeline, allowDiskUse=True))
df = pd.DataFrame(rows)
print(f"Run mode: {run_label}")
print("Loaded rows:", len(df))
df.head(3)

Using database: bigdata
Using collection: review
Total documents: 568,454
Run mode: FULL
Loaded rows: 568454


,Id,ProductId,UserId,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...


## 3) Data Cleaning and Labeling

Rules:
- Drop exact duplicate (`ProductId`, `Text`)
- Keep non-empty text
- Build sentiment labels from rating
  - `Score > 3` => positive
  - `Score < 3` => negative
  - `Score == 3` => neutral

In [103]:
required_cols = [
    "Id", "ProductId", "UserId", "Score", "Summary", "Text", "ProductURL",
    "HelpfulnessNumerator", "HelpfulnessDenominator", "Time"
]
for col in required_cols:
    if col not in df.columns:
        df[col] = np.nan

# Normalize text/string fields
for col in ["Summary", "Text", "ProductURL", "ProductId", "UserId"]:
    df[col] = df[col].fillna("").astype(str)

# Normalize numeric fields
for col in ["Score", "HelpfulnessNumerator", "HelpfulnessDenominator", "Time"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Fill missing helpfulness with 0 and clip invalid values
for col in ["HelpfulnessNumerator", "HelpfulnessDenominator"]:
    df[col] = df[col].fillna(0)
    df[col] = df[col].clip(lower=0)

# Ensure numerator <= denominator when denominator > 0
mask_invalid_help = (df["HelpfulnessDenominator"] > 0) & (df["HelpfulnessNumerator"] > df["HelpfulnessDenominator"])
df.loc[mask_invalid_help, "HelpfulnessNumerator"] = df.loc[mask_invalid_help, "HelpfulnessDenominator"]

# Remove bad rows
df = df.dropna(subset=["Score"]).copy()
df = df[df["Text"].str.strip() != ""].copy()
before = len(df)
df = df.drop_duplicates(subset=["ProductId", "Text"], keep="first").copy()
after = len(df)
print(f"Rows after dedup: {after:,} (removed {before-after:,})")

def score_to_sentiment(s):
    if s > 3:
        return "positive"
    if s < 3:
        return "negative"
    return "neutral"

df["sentiment"] = df["Score"].apply(score_to_sentiment)
print(df["sentiment"].value_counts(normalize=True).rename("ratio"))
df.head(3)

Rows after dedup: 567,130 (removed 1,324)
sentiment
positive    0.780876
negative    0.144083
neutral     0.075041
Name: ratio, dtype: float64


,Id,ProductId,UserId,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text,ProductURL,sentiment
0,1,B001E4KFG0,A3SGXH7AUHU8GW,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...,,positive
1,2,B00813GRG4,A1D87F6ZCVE5NK,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...,,negative
2,3,B000LQOCH0,ABXLMWJIXXAIN,1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...,,positive


## 4) Text Preprocessing (Strict and Explainable)

Preprocessing goals:
- Remove noisy artifacts (URL, HTML, punctuation)
- Normalize contractions and repeated whitespace
- Keep sentiment-critical words (e.g., negations such as `not`)
- Produce a clean, reproducible text feature input for TF-IDF

In [104]:
CONTRACTIONS = {
    "can't": "can not", "won't": "will not", "n't": " not",
    "'re": " are", "'s": " is", "'d": " would", "'ll": " will",
    "'t": " not", "'ve": " have", "'m": " am"
}


def expand_contractions(text: str) -> str:
    out = text
    for k, v in CONTRACTIONS.items():
        out = out.replace(k, v)
    return out


def clean_text(text: str) -> str:
    text = text.lower()
    text = expand_contractions(text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\b[a-z]\b", " ", text)  # remove isolated 1-char tokens
    text = re.sub(r"\s+", " ", text).strip()
    return text


# Keep an untouched copy for auditability
ndf = df.copy()
ndf["text_clean"] = ndf["Text"].map(clean_text)

# Quality filters after preprocessing
before_clean = len(ndf)
ndf = ndf[ndf["text_clean"].str.len() >= 3].copy()
after_clean = len(ndf)

# Add simple quality indicators for reporting
ndf["raw_len"] = ndf["Text"].str.len()
ndf["clean_len"] = ndf["text_clean"].str.len()
ndf["len_ratio"] = ndf["clean_len"] / ndf["raw_len"].replace(0, np.nan)

print(f"Rows after text cleaning: {after_clean:,} (removed {before_clean-after_clean:,})")
print(ndf[["raw_len", "clean_len", "len_ratio"]].describe().T)

# Replace working dataframe
df = ndf

Rows after text cleaning: 567,109 (removed 21)
              count        mean         std        min         25%        50%  \
raw_len    567109.0  433.916092  437.625875  12.000000  179.000000  301.00000   
clean_len  567109.0  400.477210  397.409733   3.000000  167.000000  280.00000   
len_ratio  567109.0    0.929834    0.050258   0.016393    0.919048    0.93865   

                  75%           max  
raw_len    526.000000  21409.000000  
clean_len  487.000000  20212.000000  
len_ratio    0.954248      1.051471  


## 5) Preprocessing Quality Audit

This section verifies that preprocessing is valid and report-ready:
- Class distribution after cleaning
- Missing-value status of critical fields
- Examples of raw vs cleaned text

In [105]:
critical_cols = [
    "Score", "Text", "text_clean", "sentiment", "ProductId",
    "HelpfulnessNumerator", "HelpfulnessDenominator", "Time"
]
print("Missing values in critical columns:")
print(df[critical_cols].isna().sum())

print("\nClass distribution after preprocessing:")
print(df["sentiment"].value_counts())
print(df["sentiment"].value_counts(normalize=True).round(4))

print("\nRaw vs Clean text samples:")
display(df[["Text", "text_clean", "sentiment"]].sample(5, random_state=SEED))

Missing values in critical columns:
Score                     0
Text                      0
text_clean                0
sentiment                 0
ProductId                 0
HelpfulnessNumerator      0
HelpfulnessDenominator    0
Time                      0
dtype: int64

Class distribution after preprocessing:
sentiment
positive    442837
negative     81714
neutral      42558
Name: count, dtype: int64
sentiment
positive    0.7809
negative    0.1441
neutral     0.0750
Name: proportion, dtype: float64

Raw vs Clean text samples:


,Text,text_clean,sentiment
438955,I first tasted this coffee at the Hyatt at Phi...,first tasted this coffee at the hyatt at phila...,positive
246139,Many people have referred to the fact that thi...,many people have referred to the fact that thi...,neutral
282388,I usually feed my dogs soft and chewy treats b...,usually feed my dogs soft and chewy treats but...,positive
26782,This tea is incredible!! The complexity of the...,this tea is incredible the complexity of the f...,positive
209016,"very hard to set, you could loose fingers if y...",very hard to set you could loose fingers if yo...,negative


## 6) Strict Split: Train / Validation / Test

Model selection uses train+validation only.
Final test is untouched until the very end.

In [106]:
X = df["text_clean"]
y = df["sentiment"]

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.1765, random_state=SEED, stratify=y_trainval
)
# 0.85 * 0.1765 ~= 0.15 => approx 70/15/15

print("Train:", len(X_train), "Val:", len(X_val), "Test:", len(X_test))

Train: 396961 Val: 85081 Test: 85067


## 7) Model Training with CV (No Test Leakage)

We run three strong baselines:
- MultinomialNB
- Logistic Regression
- LinearSVC

Vectorizer is inside pipeline, so each CV fold is leakage-safe.

In [107]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

experiments = {
    "nb": {
        "pipe": Pipeline([
            ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1, 2))),
            ("clf", MultinomialNB())
        ]),
        "params": {"clf__alpha": [0.1, 0.5, 1.0]}
    },
    "lr": {
        "pipe": Pipeline([
            ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1, 2))),
            ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED))
        ]),
        "params": {"clf__C": [0.1, 1, 3]}
    },
    "svm": {
        "pipe": Pipeline([
            ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1, 2))),
            ("clf", LinearSVC(class_weight="balanced", random_state=SEED))
        ]),
        "params": {"clf__C": [0.1, 1, 3]}
    }
}

best_models = {}
for name, spec in experiments.items():
    gs = GridSearchCV(
        estimator=spec["pipe"],
        param_grid=spec["params"],
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)
    best_models[name] = gs
    print(f"[{name}] best params: {gs.best_params_}, cv f1_macro: {gs.best_score_:.4f}")

[nb] best params: {'clf__alpha': 0.1}, cv f1_macro: 0.5864
[lr] best params: {'clf__C': 3}, cv f1_macro: 0.7236
[svm] best params: {'clf__C': 1}, cv f1_macro: 0.7489


## 8) Validation Comparison and Final Test

Select best model by validation macro-F1, then evaluate once on test.

In [108]:
def evaluate_model(model, X_eval, y_eval, name="model"):
    pred = model.predict(X_eval)
    acc = accuracy_score(y_eval, pred)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(y_eval, pred, average="macro", zero_division=0)
    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(y_eval, pred, average="weighted", zero_division=0)
    return {
        "model": name,
        "accuracy": acc,
        "precision_macro": p_macro,
        "recall_macro": r_macro,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
        "pred": pred
    }

val_rows = []
for name, gs in best_models.items():
    out = evaluate_model(gs.best_estimator_, X_val, y_val, name)
    val_rows.append({k: v for k, v in out.items() if k != "pred"})

val_df = pd.DataFrame(val_rows).sort_values("f1_macro", ascending=False)
display(val_df)

winner = val_df.iloc[0]["model"]
best_estimator = best_models[winner].best_estimator_
print("Selected model:", winner)

# Refit winner on train+val, then evaluate test once
best_estimator.fit(X_trainval, y_trainval)
test_pred = best_estimator.predict(X_test)

print("\n=== FINAL TEST REPORT ===")
print(classification_report(y_test, test_pred, digits=4))
print("Confusion Matrix (labels: negative, neutral, positive):")
print(confusion_matrix(y_test, test_pred, labels=["negative", "neutral", "positive"]))

,model,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted
2,svm,0.889317,0.737075,0.774178,0.753871,0.893174
1,lr,0.853880,0.695187,0.796408,0.727393,0.869347
0,nb,0.855444,0.744136,0.547406,0.586149,0.827268


Selected model: svm

=== FINAL TEST REPORT ===
              precision    recall  f1-score   support

    negative     0.7729    0.8168    0.7943     12257
     neutral     0.4841    0.5735    0.5250      6384
    positive     0.9630    0.9358    0.9492     66426

    accuracy                         0.8915     85067
   macro avg     0.7400    0.7754    0.7562     85067
weighted avg     0.8997    0.8915    0.8951     85067

Confusion Matrix (labels: negative, neutral, positive):
[[10012  1298   947]
 [ 1281  3661  1442]
 [ 1660  2603 62163]]


## 9) Business Signals from Reviews (Actionable Output)

Simple rule-based aspect flags for operational decisions.

In [109]:
aspect_keywords = {
    "delivery": ["delivery", "ship", "shipping", "arrive", "late"],
    "packaging_damage": ["broken", "damaged", "leak", "crushed", "package"],
    "price_value": ["price", "expensive", "cheap", "value", "worth"],
    "product_quality": ["quality", "taste", "flavor", "fresh", "defect"]
}

NEGATION_TOKENS = ("not", "no", "never", "without")


def keyword_hit_with_negation_guard(text, keyword):
    kw = re.escape(keyword)
    positive_pat = rf"\b{kw}\b"
    negated_pat = rf"\b(?:{'|'.join(NEGATION_TOKENS)})\s+{kw}\b"
    return bool(re.search(positive_pat, text)) and not bool(re.search(negated_pat, text))


def flag_aspects(text):
    hit = {}
    for asp, kws in aspect_keywords.items():
        hit[asp] = int(any(keyword_hit_with_negation_guard(text, kw) for kw in kws))
    return pd.Series(hit)


aspect_df = df[["ProductId", "sentiment", "text_clean"]].copy()
aspect_flags = aspect_df["text_clean"].apply(flag_aspects)
aspect_df = pd.concat([aspect_df, aspect_flags], axis=1)

neg = aspect_df[aspect_df["sentiment"] == "negative"]
summary = {
    k: float(neg[k].mean()) if len(neg) else 0.0
    for k in aspect_keywords
}
print("Negative-review aspect incidence:")
print(pd.Series(summary).sort_values(ascending=False))

Negative-review aspect incidence:
product_quality     0.389982
price_value         0.144626
packaging_damage    0.081871
delivery            0.057126
dtype: float64


## 10) Export Artifacts for Reporting (Safe Export)

These files are required for reporting evidence.
If a target CSV is locked/opened by another app, fallback filenames are used automatically.

In [110]:
def safe_to_csv(df_obj, primary_name):
    try:
        df_obj.to_csv(primary_name, index=False)
        print(f"Saved: {primary_name}")
    except PermissionError:
        fallback = primary_name.replace('.csv', '_new.csv')
        df_obj.to_csv(fallback, index=False)
        print(f"PermissionError on {primary_name}; saved fallback: {fallback}")

safe_to_csv(val_df, "strict_model_validation_results.csv")
safe_to_csv(pd.DataFrame({"y_true": y_test.values, "y_pred": test_pred}), "strict_test_predictions.csv")
safe_to_csv(df[["Id", "ProductId", "Score", "sentiment", "text_clean"]], "strict_cleaned_full.csv")

print("Export step completed.")

Saved: strict_model_validation_results.csv
Saved: strict_test_predictions.csv
Saved: strict_cleaned_full.csv
Export step completed.


## 11) Credibility Scoring with Helpfulness (Wilson Lower Bound)

Why this matters for business:
- Reviews with few votes are noisy.
- We assign each review a credibility score using Wilson lower bound.
- Product-level decisions then prioritize signals with stronger evidence.

In [111]:
# Wilson lower bound for Bernoulli proportion p = helpful_num / helpful_den
# z=1.96 => 95% confidence

def wilson_lower_bound(pos, n, z=1.96):
    if n <= 0:
        return 0.0
    phat = pos / n
    denominator = 1 + z**2 / n
    numerator = phat + z**2/(2*n) - z * np.sqrt((phat*(1-phat) + z**2/(4*n)) / n)
    return numerator / denominator


df["helpful_num"] = df["HelpfulnessNumerator"].astype(float)
df["helpful_den"] = df["HelpfulnessDenominator"].astype(float)
df["helpful_ratio"] = np.where(df["helpful_den"] > 0, df["helpful_num"] / df["helpful_den"], 0.0)

df["review_credibility"] = [
    wilson_lower_bound(p, n)
    for p, n in zip(df["helpful_num"].values, df["helpful_den"].values)
]

print(df[["helpful_num", "helpful_den", "helpful_ratio", "review_credibility"]].describe().T)

                       count      mean       std           min  25%  50%  \
helpful_num         567109.0  1.741371  7.640057  0.000000e+00  0.0  0.0   
helpful_den         567109.0  2.224371  8.289614  0.000000e+00  0.0  1.0   
helpful_ratio       567109.0  0.407787  0.462095  0.000000e+00  0.0  0.0   
review_credibility  567109.0  0.146894  0.200552 -2.209651e-17  0.0  0.0   

                         75%         max  
helpful_num         2.000000  866.000000  
helpful_den         2.000000  923.000000  
helpful_ratio       1.000000    1.000000  
review_credibility  0.206543    0.984424  


## 12) Time Trend and Early Warning Signals

We convert Unix `Time` to calendar month and monitor negative sentiment trend.
A rising short-term negative rate vs long-term baseline triggers early-warning.

In [112]:
# Time conversion (dataset has Unix timestamp in seconds)
df["review_datetime"] = pd.to_datetime(df["Time"], unit="s", errors="coerce")
df["review_month"] = df["review_datetime"].dt.to_period("M").astype(str)

# Build monthly trend table
time_df = df.dropna(subset=["review_datetime"]).copy()
time_df["is_negative"] = (time_df["sentiment"] == "negative").astype(int)

monthly = (
    time_df
    .groupby("review_month")
    .agg(
        reviews=("Id", "count"),
        neg_rate=("is_negative", "mean"),
        mean_score=("Score", "mean")
    )
    .reset_index()
    .sort_values("review_month")
)

monthly["neg_rate_3m"] = monthly["neg_rate"].rolling(3, min_periods=1).mean()
monthly["neg_rate_12m"] = monthly["neg_rate"].rolling(12, min_periods=3).mean()
monthly["early_warning"] = (monthly["neg_rate_3m"] - monthly["neg_rate_12m"]) > 0.02

print("Latest 12 months trend:")
display(monthly.tail(12))

latest_warning = bool(monthly["early_warning"].iloc[-1]) if len(monthly) else False
print("Global early-warning status:", latest_warning)

Latest 12 months trend:


,review_month,reviews,neg_rate,mean_score,neg_rate_3m,neg_rate_12m,early_warning
131,2011-11,16217,0.153604,4.133625,0.149187,0.151940,False
132,2011-12,19035,0.172367,4.080273,0.159857,0.152162,False
133,2012-01,21682,0.171525,4.095794,0.165832,0.152707,False
134,2012-02,18794,0.160902,4.105300,0.168265,0.152291,False
135,2012-03,19707,0.165068,4.071548,0.165832,0.154449,False
136,2012-04,18897,0.153781,4.126581,0.159917,0.155443,False
137,2012-05,19396,0.154413,4.119509,0.157754,0.156538,False
138,2012-06,18089,0.161092,4.085024,0.156429,0.156526,False
139,2012-07,19835,0.149836,4.147719,0.155114,0.156536,False
140,2012-08,19798,0.156935,4.124053,0.155955,0.157790,False


Global early-warning status: False


## 13) Review Intelligence Scoring and Product Decision Engine

This block generates a decision-ready product table:
- Combines sentiment, volume, credibility, and complaint patterns
- Produces recommended actions: INVEST / FIX / KILL-OR-REDESIGN / MONITOR

In [132]:
# Rebuild aspect flags with clean text for product-level scoring
# Reuse the same aspect rules defined earlier for consistency.
prod_base = df[[
    "Id", "ProductId", "Score", "sentiment", "text_clean", "review_credibility", "review_datetime"
]].copy()
prod_flags = prod_base["text_clean"].apply(flag_aspects)
prod_df = pd.concat([prod_base, prod_flags], axis=1)

# sentiment indicator columns
prod_df["is_pos"] = (prod_df["sentiment"] == "positive").astype(int)
prod_df["is_neg"] = (prod_df["sentiment"] == "negative").astype(int)
prod_df["is_neu"] = (prod_df["sentiment"] == "neutral").astype(int)

# Weighted sentiment score in [-1, 1] using review credibility
sent_map = {"negative": -1.0, "neutral": 0.0, "positive": 1.0}
prod_df["sent_score"] = prod_df["sentiment"].map(sent_map)
prod_df["weighted_sent"] = prod_df["sent_score"] * prod_df["review_credibility"]

# Aggregate by product
product_metrics = (
    prod_df
    .groupby("ProductId")
    .agg(
        review_count=("Id", "count"),
        mean_score=("Score", "mean"),
        pos_rate=("is_pos", "mean"),
        neg_rate=("is_neg", "mean"),
        neu_rate=("is_neu", "mean"),
        avg_credibility=("review_credibility", "mean"),
        weighted_sentiment=("weighted_sent", "mean"),
        delivery_rate=("delivery", "mean"),
        packaging_damage_rate=("packaging_damage", "mean"),
        price_value_rate=("price_value", "mean"),
        product_quality_rate=("product_quality", "mean"),
        last_review_date=("review_datetime", "max")
    )
    .reset_index()
)

# Risk and opportunity scores (heuristic but explicit)
# MIN_REVIEWS guard prevents over-confident actions on very small samples.
MIN_REVIEWS = 30

# Demand proxy quantiles are computed on reliable products only.
# This avoids distorted thresholds caused by long-tail products with 1-2 reviews.
reliable_products = product_metrics[product_metrics["review_count"] >= MIN_REVIEWS]
if len(reliable_products) > 0:
    q75 = reliable_products["review_count"].quantile(0.75)
    q25 = reliable_products["review_count"].quantile(0.25)
else:
    q75 = product_metrics["review_count"].quantile(0.75)
    q25 = product_metrics["review_count"].quantile(0.25)

product_metrics["demand_segment"] = np.select(
    [product_metrics["review_count"] >= q75, product_metrics["review_count"] <= q25],
    ["high", "low"],
    default="mid"
)

base_risk = (
    0.50 * product_metrics["neg_rate"] +
    0.25 * product_metrics["packaging_damage_rate"] +
    0.20 * product_metrics["delivery_rate"] +
    0.05 * product_metrics["avg_credibility"]
)

# Confidence ramps from 0->1 as review_count approaches MIN_REVIEWS.
confidence = (product_metrics["review_count"] / MIN_REVIEWS).clip(upper=1.0)
product_metrics["risk_score"] = base_risk * confidence

# Log-scaled review volume gives fairer signal under long-tail distributions.
review_volume_score = np.log1p(product_metrics["review_count"]) / np.log1p(product_metrics["review_count"].max())

product_metrics["opportunity_score"] = (
    0.45 * product_metrics["pos_rate"] +
    0.25 * product_metrics["weighted_sentiment"].clip(-1, 1).add(1).div(2) +
    0.20 * review_volume_score +
    0.10 * product_metrics["avg_credibility"]
)

# Tunable thresholds for explainability and easier calibration.
INVEST_OPP_MIN = 0.62
INVEST_RISK_MAX = 0.22
KILL_RISK_MIN = 0.34
KILL_OPP_MAX = 0.45
FIX_RISK_MIN = 0.30

# Guard against over-aggressive FIX assignment for clearly healthy products.
HEALTHY_SCORE_MIN = 4.2
HEALTHY_POS_MIN = 0.80
HEALTHY_NEG_MAX = 0.15
healthy_signal = (
    (product_metrics["mean_score"] >= HEALTHY_SCORE_MIN) &
    (product_metrics["pos_rate"] >= HEALTHY_POS_MIN) &
    (product_metrics["neg_rate"] < HEALTHY_NEG_MAX)
)

# Decision engine rules
conditions = [
    # INVEST: strong sentiment + sufficient demand + low risk + enough evidence
    (
        (product_metrics["review_count"] >= MIN_REVIEWS) &
        (product_metrics["opportunity_score"] >= INVEST_OPP_MIN) &
        (product_metrics["risk_score"] <= INVEST_RISK_MAX) &
        (product_metrics["review_count"] >= q75)
    ),
    # KILL/REDESIGN: high risk + weak opportunity + low demand + enough evidence
    (
        (product_metrics["review_count"] >= MIN_REVIEWS) &
        (product_metrics["risk_score"] >= KILL_RISK_MIN) &
        (product_metrics["opportunity_score"] <= KILL_OPP_MAX) &
        (product_metrics["review_count"] <= q25)
    ),
    # FIX: elevated risk with sufficient evidence, excluding clearly healthy products
    (
        (product_metrics["review_count"] >= MIN_REVIEWS) &
        (product_metrics["risk_score"] >= FIX_RISK_MIN) &
        (product_metrics["review_count"] > q25) &
        (~healthy_signal)
    )
]
choices = ["INVEST", "KILL_OR_REDESIGN", "FIX"]
product_metrics["recommended_action"] = np.select(conditions, choices, default="MONITOR")

# Priority score to rank action queue
action_weight = {
    "FIX": 1.00,
    "KILL_OR_REDESIGN": 0.95,
    "INVEST": 0.80,
    "MONITOR": 0.30
}
product_metrics["action_priority"] = product_metrics.apply(
    lambda r: action_weight[r["recommended_action"]] * (r["risk_score"] + r["opportunity_score"]),
    axis=1
)

decision_table = product_metrics.sort_values("action_priority", ascending=False).reset_index(drop=True)

print(f"MIN_REVIEWS threshold: {MIN_REVIEWS}")
print("Products below MIN_REVIEWS:", int((decision_table["review_count"] < MIN_REVIEWS).sum()))
print("Action distribution:")
print(decision_table["recommended_action"].value_counts())
print("\nTop 20 priority actions:")
display(decision_table.head(20))

MIN_REVIEWS threshold: 30
Products below MIN_REVIEWS: 71127
Action distribution:
recommended_action
MONITOR             73698
INVEST                534
FIX                    17
KILL_OR_REDESIGN        9
Name: count, dtype: int64

Top 20 priority actions:


,ProductId,review_count,mean_score,pos_rate,neg_rate,neu_rate,avg_credibility,weighted_sentiment,delivery_rate,packaging_damage_rate,price_value_rate,product_quality_rate,last_review_date,demand_segment,risk_score,opportunity_score,recommended_action,action_priority
0,B000AQJRWG,79,3.278481,0.556962,0.379747,0.063291,0.351463,0.162706,0.303797,0.253165,0.139241,0.316456,2012-04-19,mid,0.331497,0.559726,FIX,0.891223
1,B0002DH5MG,43,1.372093,0.069767,0.906977,0.023256,0.382580,-0.323957,0.023256,0.139535,0.023256,0.000000,2012-10-03,mid,0.512152,0.265221,FIX,0.777373
2,B005UP1M4I,42,2.500000,0.357143,0.595238,0.047619,0.298565,-0.097768,0.142857,0.047619,0.285714,0.166667,2012-10-08,mid,0.353024,0.413737,FIX,0.766761
3,B006N3I69A,131,1.343511,0.038168,0.931298,0.030534,0.188337,-0.180193,0.000000,0.038168,0.221374,0.274809,2012-09-28,high,0.484608,0.281791,FIX,0.766398
4,B004ZU0RAE,44,2.431818,0.318182,0.590909,0.090909,0.034014,0.019932,0.022727,0.272727,0.250000,0.022727,2012-10-16,mid,0.369883,0.385796,FIX,0.755679
5,B004H3N2LU,83,1.373494,0.084337,0.903614,0.012048,0.100623,-0.088016,0.012048,0.012048,0.180723,0.337349,2012-08-08,mid,0.462260,0.292052,FIX,0.754312
6,B0085MLY5A,82,2.439024,0.317073,0.658537,0.024390,0.210046,-0.130628,0.012195,0.036585,0.073171,0.634146,2012-10-19,mid,0.351356,0.402048,FIX,0.753404
7,B005BHIESC,82,2.439024,0.317073,0.658537,0.024390,0.210046,-0.130628,0.012195,0.036585,0.073171,0.634146,2012-10-19,mid,0.351356,0.402048,FIX,0.753404
8,B000X1Q1G8,184,1.695652,0.141304,0.766304,0.092391,0.222740,-0.149017,0.021739,0.010870,0.086957,0.565217,2012-10-16,high,0.401354,0.345446,FIX,0.746800
9,B0030GOO5S,44,2.568182,0.409091,0.568182,0.022727,0.170360,-0.020962,0.022727,0.045455,0.181818,0.454545,2012-08-10,mid,0.308518,0.435228,FIX,0.743746


## 13.5) Business-Friendly Product Grouping (Best Seller / Hidden Gem / Problem)

To make product strategy easier to explain, we add explicit business group labels on top of model-driven scores:
- `best_seller`: high score and high review volume
- `hidden_gem`: high score but low review volume
- `problem_high_volume`: low score but high review volume
- `other`: remaining products

In [ ]:
# Add human-readable business groups for presentation and decision discussion
vol_high = decision_table["review_count"].quantile(0.75)
vol_low = decision_table["review_count"].quantile(0.25)
score_high = 4.2
score_low = 3.2

conditions_group = [
    (decision_table["mean_score"] >= score_high) & (decision_table["review_count"] >= vol_high),
    (decision_table["mean_score"] >= score_high) & (decision_table["review_count"] <= vol_low),
    (decision_table["mean_score"] <= score_low) & (decision_table["review_count"] >= vol_high),
]
choices_group = ["best_seller", "hidden_gem", "problem_high_volume"]

decision_table["business_group"] = np.select(conditions_group, choices_group, default="other")

print("Business group distribution:")
print(decision_table["business_group"].value_counts())

print("\nTop products by business group (preview):")
for grp in ["best_seller", "hidden_gem", "problem_high_volume"]:
    print(f"\n[{grp}]")
    
    if grp in ["best_seller", "hidden_gem"]:
        sort_metric = "opportunity_score"
    else:
        sort_metric = "risk_score"

    display(
        decision_table[decision_table["business_group"] == grp][
            ["ProductId", "review_count", "mean_score", "risk_score", "opportunity_score", "recommended_action", "action_priority"]
        ].sort_values(sort_metric, ascending=False).head(10)
    )

Business group distribution:
business_group
other                  41183
hidden_gem             19513
best_seller            12072
problem_high_volume     1490
Name: count, dtype: int64

Top products by business group (preview):

[best_seller]


,ProductId,review_count,mean_score,risk_score,opportunity_score,recommended_action,action_priority
23,B000VK08OC,232,4.491379,0.081944,0.774601,INVEST,0.685236
26,B003B3OOPA,623,4.739968,0.036720,0.790643,INVEST,0.661891
27,B001EO5Q64,567,4.746032,0.042959,0.782191,INVEST,0.660120
28,B008J1HO4C,355,4.681690,0.054662,0.762178,INVEST,0.653472
29,B001EO5U3I,355,4.681690,0.054662,0.762178,INVEST,0.653472
31,B000GAT6NG,389,4.802057,0.040723,0.775001,INVEST,0.652579
32,B004EAGP74,389,4.802057,0.040723,0.775001,INVEST,0.652579
33,B001E8DHPW,389,4.802057,0.040723,0.775001,INVEST,0.652579
36,B000CPZSC8,133,4.774436,0.093402,0.720652,INVEST,0.651243
38,B000EDG3UE,114,4.508772,0.099037,0.711095,INVEST,0.648106



[hidden_gem]


,ProductId,review_count,mean_score,risk_score,opportunity_score,recommended_action,action_priority
723,B000F7A5WQ,1,5.0,0.001428,0.788141,MONITOR,0.236871
739,B0040IZMY2,1,5.0,0.008026,0.778870,MONITOR,0.236069
786,B000AP58O8,1,5.0,0.007994,0.774468,MONITOR,0.234738
805,B004EBPTIO,1,5.0,0.009634,0.770922,MONITOR,0.234167
806,B0060OPGKE,1,5.0,0.009634,0.770922,MONITOR,0.234167
837,B003J9UZUE,1,5.0,0.001352,0.777834,MONITOR,0.233756
856,B004DGU7G4,1,5.0,0.001344,0.776780,MONITOR,0.233437
857,B0000DZEV2,1,5.0,0.001344,0.776780,MONITOR,0.233437
858,B0002I9S26,1,5.0,0.001344,0.776780,MONITOR,0.233437
859,B001EPPGXG,1,5.0,0.001344,0.776780,MONITOR,0.233437



[problem_high_volume]


,ProductId,review_count,mean_score,risk_score,opportunity_score,recommended_action,action_priority
1,B0002DH5MG,43,1.372093,0.512152,0.265221,FIX,0.777373
2,B005UP1M4I,42,2.500000,0.353024,0.413737,FIX,0.766761
3,B006N3I69A,131,1.343511,0.484608,0.281791,FIX,0.766398
4,B004ZU0RAE,44,2.431818,0.369883,0.385796,FIX,0.755679
5,B004H3N2LU,83,1.373494,0.462260,0.292052,FIX,0.754312
6,B0085MLY5A,82,2.439024,0.351356,0.402048,FIX,0.753404
7,B005BHIESC,82,2.439024,0.351356,0.402048,FIX,0.753404
8,B000X1Q1G8,184,1.695652,0.401354,0.345446,FIX,0.746800
9,B0030GOO5S,44,2.568182,0.308518,0.435228,FIX,0.743746
10,B000ER5CTS,65,2.738462,0.307447,0.434363,FIX,0.741810


## 14) Final Business Dashboard Table (Executive Summary)

This table is the final delivery for decision-making:
- one row per product
- risk and opportunity context
- explicit action recommendation

In [134]:
dashboard_cols = [
    "ProductId", "review_count", "mean_score",
    "pos_rate", "neg_rate", "avg_credibility",
    "delivery_rate", "packaging_damage_rate", "price_value_rate", "product_quality_rate",
    "risk_score", "opportunity_score", "business_group", "recommended_action", "action_priority", "last_review_date"
]

decision_dashboard = decision_table[dashboard_cols].copy()

decision_dashboard = decision_dashboard.sort_values(
    ["recommended_action", "action_priority"],
    ascending=[True, False]
).reset_index(drop=True)

# Guardrail check: low-review products must stay MONITOR.
MIN_REVIEWS = 30  # keep consistent with decision engine
check_low = decision_dashboard[decision_dashboard["review_count"] < MIN_REVIEWS].copy()

print("Rows with review_count < MIN_REVIEWS:", len(check_low))
print("\nAction distribution for low-review rows:")
print(check_low["recommended_action"].value_counts())

invalid = check_low[check_low["recommended_action"].isin(["INVEST", "FIX", "KILL_OR_REDESIGN"])]
print("\nInvalid low-review actions (should be 0):", len(invalid))

display(check_low.head(20))

# Sanity check: FIX candidates that still look strongly healthy.
suspicious_fix = decision_dashboard[
    (decision_dashboard["recommended_action"] == "FIX") &
    (decision_dashboard["mean_score"] >= 4.2) &
    (decision_dashboard["pos_rate"] >= 0.80) &
    (decision_dashboard["neg_rate"] < 0.15)
].copy()
print("\nSuspicious FIX products:", len(suspicious_fix))
if len(suspicious_fix) > 0:
    display(suspicious_fix.head(20))

print("Executive dashboard preview:")
display(decision_dashboard.head(30))

# Action-specific slices for reporting
invest_top = decision_dashboard[decision_dashboard["recommended_action"] == "INVEST"].head(20)
fix_top = decision_dashboard[decision_dashboard["recommended_action"] == "FIX"].head(20)
kill_top = decision_dashboard[decision_dashboard["recommended_action"] == "KILL_OR_REDESIGN"].head(20)

print("Top INVEST products:")
display(invest_top)
print("Top FIX products:")
display(fix_top)
print("Top KILL_OR_REDESIGN products:")
display(kill_top)

Rows with review_count < MIN_REVIEWS: 71127

Action distribution for low-review rows:
recommended_action
MONITOR    71127
Name: count, dtype: int64

Invalid low-review actions (should be 0): 0


,ProductId,review_count,mean_score,pos_rate,neg_rate,avg_credibility,delivery_rate,packaging_damage_rate,price_value_rate,product_quality_rate,risk_score,opportunity_score,business_group,recommended_action,action_priority,last_review_date
561,B009K2BBQG,10,4.900000,1.000000,0.000000,0.843636,0.000000,0.100000,0.500000,0.100000,0.022394,0.835194,best_seller,MONITOR,0.257276,2011-11-29
562,B0065G59F4,10,4.900000,1.000000,0.000000,0.843636,0.000000,0.100000,0.500000,0.100000,0.022394,0.835194,best_seller,MONITOR,0.257276,2011-11-29
563,B009K2BBT8,10,4.900000,1.000000,0.000000,0.843636,0.000000,0.100000,0.500000,0.100000,0.022394,0.835194,best_seller,MONITOR,0.257276,2011-11-29
564,B008VQ01Y0,10,4.900000,1.000000,0.000000,0.843636,0.000000,0.100000,0.500000,0.100000,0.022394,0.835194,best_seller,MONITOR,0.257276,2011-11-29
565,B0069GOKGE,10,4.900000,1.000000,0.000000,0.843636,0.000000,0.100000,0.500000,0.100000,0.022394,0.835194,best_seller,MONITOR,0.257276,2011-11-29
566,B00859FH7O,10,4.900000,1.000000,0.000000,0.843636,0.000000,0.100000,0.500000,0.100000,0.022394,0.835194,best_seller,MONITOR,0.257276,2011-11-29
567,B0051SU0OW,10,4.900000,1.000000,0.000000,0.843636,0.000000,0.100000,0.500000,0.100000,0.022394,0.835194,best_seller,MONITOR,0.257276,2011-11-29
568,B005FKRTO0,10,4.900000,1.000000,0.000000,0.843636,0.000000,0.100000,0.500000,0.100000,0.022394,0.835194,best_seller,MONITOR,0.257276,2011-11-29
569,B005F3IAPE,10,4.900000,1.000000,0.000000,0.843636,0.000000,0.100000,0.500000,0.100000,0.022394,0.835194,best_seller,MONITOR,0.257276,2011-11-29
570,B008F2K1YU,10,4.900000,1.000000,0.000000,0.843636,0.000000,0.100000,0.500000,0.100000,0.022394,0.835194,best_seller,MONITOR,0.257276,2011-11-29



Suspicious FIX products: 0
Executive dashboard preview:


,ProductId,review_count,mean_score,pos_rate,neg_rate,avg_credibility,delivery_rate,packaging_damage_rate,price_value_rate,product_quality_rate,risk_score,opportunity_score,business_group,recommended_action,action_priority,last_review_date
0,B000AQJRWG,79,3.278481,0.556962,0.379747,0.351463,0.303797,0.253165,0.139241,0.316456,0.331497,0.559726,other,FIX,0.891223,2012-04-19
1,B0002DH5MG,43,1.372093,0.069767,0.906977,0.382580,0.023256,0.139535,0.023256,0.000000,0.512152,0.265221,problem_high_volume,FIX,0.777373,2012-10-03
2,B005UP1M4I,42,2.500000,0.357143,0.595238,0.298565,0.142857,0.047619,0.285714,0.166667,0.353024,0.413737,problem_high_volume,FIX,0.766761,2012-10-08
3,B006N3I69A,131,1.343511,0.038168,0.931298,0.188337,0.000000,0.038168,0.221374,0.274809,0.484608,0.281791,problem_high_volume,FIX,0.766398,2012-09-28
4,B004ZU0RAE,44,2.431818,0.318182,0.590909,0.034014,0.022727,0.272727,0.250000,0.022727,0.369883,0.385796,problem_high_volume,FIX,0.755679,2012-10-16
5,B004H3N2LU,83,1.373494,0.084337,0.903614,0.100623,0.012048,0.012048,0.180723,0.337349,0.462260,0.292052,problem_high_volume,FIX,0.754312,2012-08-08
6,B0085MLY5A,82,2.439024,0.317073,0.658537,0.210046,0.012195,0.036585,0.073171,0.634146,0.351356,0.402048,problem_high_volume,FIX,0.753404,2012-10-19
7,B005BHIESC,82,2.439024,0.317073,0.658537,0.210046,0.012195,0.036585,0.073171,0.634146,0.351356,0.402048,problem_high_volume,FIX,0.753404,2012-10-19
8,B000X1Q1G8,184,1.695652,0.141304,0.766304,0.222740,0.021739,0.010870,0.086957,0.565217,0.401354,0.345446,problem_high_volume,FIX,0.746800,2012-10-16
9,B0030GOO5S,44,2.568182,0.409091,0.568182,0.170360,0.022727,0.045455,0.181818,0.454545,0.308518,0.435228,problem_high_volume,FIX,0.743746,2012-08-10


Top INVEST products:


,ProductId,review_count,mean_score,pos_rate,neg_rate,avg_credibility,delivery_rate,packaging_damage_rate,price_value_rate,product_quality_rate,risk_score,opportunity_score,business_group,recommended_action,action_priority,last_review_date
17,B000VK08OC,232,4.491379,0.870690,0.077586,0.470784,0.081897,0.012931,0.258621,0.327586,0.081944,0.774601,best_seller,INVEST,0.685236,2012-10-25
18,B003B3OOPA,623,4.739968,0.939005,0.020867,0.248043,0.043339,0.020867,0.179775,0.296950,0.036720,0.790643,best_seller,INVEST,0.661891,2012-10-26
19,B001EO5Q64,567,4.746032,0.950617,0.022928,0.199571,0.072310,0.028219,0.208113,0.446208,0.042959,0.782191,best_seller,INVEST,0.660120,2012-10-23
20,B008J1HO4C,355,4.681690,0.926761,0.033803,0.228444,0.092958,0.030986,0.295775,0.408451,0.054662,0.762178,best_seller,INVEST,0.653472,2012-10-25
21,B001EO5U3I,355,4.681690,0.926761,0.033803,0.228444,0.092958,0.030986,0.295775,0.408451,0.054662,0.762178,best_seller,INVEST,0.653472,2012-10-25
22,B000FI4O90,190,3.894737,0.705263,0.210526,0.427080,0.084211,0.047368,0.152632,0.389474,0.155301,0.661243,other,INVEST,0.653235,2012-10-24
23,B000GAT6NG,389,4.802057,0.953728,0.012853,0.207768,0.100257,0.015424,0.241645,0.408740,0.040723,0.775001,best_seller,INVEST,0.652579,2012-10-26
24,B004EAGP74,389,4.802057,0.953728,0.012853,0.207768,0.100257,0.015424,0.241645,0.408740,0.040723,0.775001,best_seller,INVEST,0.652579,2012-10-26
25,B001E8DHPW,389,4.802057,0.953728,0.012853,0.207768,0.100257,0.015424,0.241645,0.408740,0.040723,0.775001,best_seller,INVEST,0.652579,2012-10-26
26,B000HDI5O8,254,4.031496,0.732283,0.185039,0.193269,0.181102,0.129921,0.212598,0.259843,0.170884,0.643515,other,INVEST,0.651519,2012-10-20


Top FIX products:


,ProductId,review_count,mean_score,pos_rate,neg_rate,avg_credibility,delivery_rate,packaging_damage_rate,price_value_rate,product_quality_rate,risk_score,opportunity_score,business_group,recommended_action,action_priority,last_review_date
0,B000AQJRWG,79,3.278481,0.556962,0.379747,0.351463,0.303797,0.253165,0.139241,0.316456,0.331497,0.559726,other,FIX,0.891223,2012-04-19
1,B0002DH5MG,43,1.372093,0.069767,0.906977,0.382580,0.023256,0.139535,0.023256,0.000000,0.512152,0.265221,problem_high_volume,FIX,0.777373,2012-10-03
2,B005UP1M4I,42,2.500000,0.357143,0.595238,0.298565,0.142857,0.047619,0.285714,0.166667,0.353024,0.413737,problem_high_volume,FIX,0.766761,2012-10-08
3,B006N3I69A,131,1.343511,0.038168,0.931298,0.188337,0.000000,0.038168,0.221374,0.274809,0.484608,0.281791,problem_high_volume,FIX,0.766398,2012-09-28
4,B004ZU0RAE,44,2.431818,0.318182,0.590909,0.034014,0.022727,0.272727,0.250000,0.022727,0.369883,0.385796,problem_high_volume,FIX,0.755679,2012-10-16
5,B004H3N2LU,83,1.373494,0.084337,0.903614,0.100623,0.012048,0.012048,0.180723,0.337349,0.462260,0.292052,problem_high_volume,FIX,0.754312,2012-08-08
6,B0085MLY5A,82,2.439024,0.317073,0.658537,0.210046,0.012195,0.036585,0.073171,0.634146,0.351356,0.402048,problem_high_volume,FIX,0.753404,2012-10-19
7,B005BHIESC,82,2.439024,0.317073,0.658537,0.210046,0.012195,0.036585,0.073171,0.634146,0.351356,0.402048,problem_high_volume,FIX,0.753404,2012-10-19
8,B000X1Q1G8,184,1.695652,0.141304,0.766304,0.222740,0.021739,0.010870,0.086957,0.565217,0.401354,0.345446,problem_high_volume,FIX,0.746800,2012-10-16
9,B0030GOO5S,44,2.568182,0.409091,0.568182,0.170360,0.022727,0.045455,0.181818,0.454545,0.308518,0.435228,problem_high_volume,FIX,0.743746,2012-08-10


Top KILL_OR_REDESIGN products:


,ProductId,review_count,mean_score,pos_rate,neg_rate,avg_credibility,delivery_rate,packaging_damage_rate,price_value_rate,product_quality_rate,risk_score,opportunity_score,business_group,recommended_action,action_priority,last_review_date
551,B00171417E,32,2.718750,0.375000,0.531250,0.346008,0.156250,0.125000,0.187500,0.437500,0.345425,0.417444,problem_high_volume,KILL_OR_REDESIGN,0.724726,2012-09-11
552,B00171417O,32,2.718750,0.375000,0.531250,0.346008,0.156250,0.125000,0.187500,0.437500,0.345425,0.417444,problem_high_volume,KILL_OR_REDESIGN,0.724726,2012-09-11
553,B000NY8OLK,38,1.315789,0.052632,0.947368,0.446890,0.000000,0.052632,0.236842,0.421053,0.509187,0.247019,problem_high_volume,KILL_OR_REDESIGN,0.718395,2012-03-10
554,B000BZ6LXO,36,2.305556,0.305556,0.666667,0.264785,0.000000,0.055556,0.055556,0.083333,0.360461,0.380462,problem_high_volume,KILL_OR_REDESIGN,0.703877,2012-09-05
555,B0099HD3YA,34,2.117647,0.235294,0.676471,0.492138,0.147059,0.088235,0.294118,0.147059,0.414313,0.325870,problem_high_volume,KILL_OR_REDESIGN,0.703173,2012-10-26
556,B003VXFK3U,31,2.548387,0.322581,0.548387,0.142355,0.032258,0.225806,0.322581,0.451613,0.344215,0.379394,problem_high_volume,KILL_OR_REDESIGN,0.687428,2012-10-20
557,B004Z33GI6,35,1.828571,0.171429,0.771429,0.124644,0.000000,0.000000,0.142857,0.600000,0.391946,0.312255,problem_high_volume,KILL_OR_REDESIGN,0.668992,2012-10-03
558,B00002N8SM,38,1.789474,0.157895,0.763158,0.041722,0.078947,0.000000,0.078947,0.000000,0.399455,0.302842,problem_high_volume,KILL_OR_REDESIGN,0.667182,2012-09-28
559,B000MICPUI,32,1.906250,0.187500,0.718750,0.173414,0.000000,0.000000,0.156250,0.437500,0.368046,0.315600,problem_high_volume,KILL_OR_REDESIGN,0.649464,2012-10-18


## 15) Export Final Decision Artifacts

Exports include model outputs and decision-engine outputs for slide/report evidence.

In [135]:
monthly.to_csv("strict_monthly_trend.csv", index=False)
decision_table.to_csv("strict_decision_table_full.csv", index=False)
decision_dashboard.to_csv("strict_decision_dashboard.csv", index=False)
invest_top.to_csv("strict_top_invest.csv", index=False)
fix_top.to_csv("strict_top_fix.csv", index=False)
kill_top.to_csv("strict_top_kill_or_redesign.csv", index=False)

print("Exported additional files:")
print("- strict_monthly_trend.csv")
print("- strict_decision_table_full.csv")
print("- strict_decision_dashboard.csv")
print("- strict_top_invest.csv")
print("- strict_top_fix.csv")
print("- strict_top_kill_or_redesign.csv")

Exported additional files:
- strict_monthly_trend.csv
- strict_decision_table_full.csv
- strict_decision_dashboard.csv
- strict_top_invest.csv
- strict_top_fix.csv
- strict_top_kill_or_redesign.csv


## 16) Data Validation and Quality Assurance (Professor Checklist)

This section proves preprocessing quality beyond model metrics:
1. Row count before/after cleaning
2. Missing-value report
3. Duplicate removal count
4. Score/date range validation
5. Label consistency validation

In [136]:
# Reconstruct key row-count checkpoints
raw_rows = len(rows)  # rows loaded from MongoDB pipeline
post_basic_clean_rows = after
post_text_clean_rows = len(df)

print("Raw rows:", raw_rows)
print("After basic cleaning (dedup/non-empty):", post_basic_clean_rows)
print("After text preprocessing:", post_text_clean_rows)
print("Duplicate rows removed:", raw_rows - post_basic_clean_rows)

print("\nMissing values (%):")
print((df.isnull().mean() * 100).sort_values(ascending=False).head(20))

print("\nScore range:", float(df["Score"].min()), float(df["Score"].max()))
print("Time range (unix):", int(df["Time"].min()), int(df["Time"].max()))

if "review_datetime" in df.columns:
    print("Review date range:", df["review_datetime"].min(), "->", df["review_datetime"].max())

print("\nLabel consistency (sentiment vs score):")
print(df.groupby("sentiment")["Score"].value_counts().sort_index())

wrong_label = df[
    ((df["Score"] > 3) & (df["sentiment"] != "positive")) |
    ((df["Score"] < 3) & (df["sentiment"] != "negative")) |
    ((df["Score"] == 3) & (df["sentiment"] != "neutral"))
].copy()
print("\nWrong sentiment labels:", len(wrong_label))
if len(wrong_label) > 0:
    display(wrong_label[["Id", "Score", "sentiment", "Text"]].head(10))

Raw rows: 568454
After basic cleaning (dedup/non-empty): 567130
After text preprocessing: 567109
Duplicate rows removed: 1324

Missing values (%):
Id                        0.0
ProductId                 0.0
UserId                    0.0
HelpfulnessNumerator      0.0
HelpfulnessDenominator    0.0
Score                     0.0
Time                      0.0
Summary                   0.0
Text                      0.0
ProductURL                0.0
sentiment                 0.0
text_clean                0.0
raw_len                   0.0
clean_len                 0.0
len_ratio                 0.0
helpful_num               0.0
helpful_den               0.0
helpful_ratio             0.0
review_credibility        0.0
review_datetime           0.0
dtype: float64

Score range: 1.0 5.0
Time range (unix): 939340800 1351209600
Review date range: 1999-10-08 00:00:00 -> 2012-10-26 00:00:00

Label consistency (sentiment vs score):
sentiment  Score
negative   1         51962
           2         29752
ne

## 17) NLP Evidence (Word Counting + Top Terms)

This satisfies the assignment requirement to count word occurrence and extract textual information.

In [137]:
from sklearn.feature_extraction.text import CountVectorizer

# 1) Target-word counting demo
target_word = "fresh"
word_count = df["text_clean"].str.count(rf"\b{target_word}\b").sum()
print(f"The word '{target_word}' appears {int(word_count):,} times.")

# 2) Top frequent words
cv_demo = CountVectorizer(stop_words="english", max_features=30)
X_demo = cv_demo.fit_transform(df["text_clean"])
word_freq = pd.DataFrame({
    "word": cv_demo.get_feature_names_out(),
    "count": np.asarray(X_demo.sum(axis=0)).ravel()
}).sort_values("count", ascending=False)

print("\nTop frequent terms:")
display(word_freq.head(20))

The word 'fresh' appears 30,239 times.

Top frequent terms:


,word,count
16,like,255158
13,good,199491
23,taste,172242
15,just,171492
14,great,166550
5,coffee,164426
21,product,151191
11,flavor,147362
24,tea,136058
18,love,127098


## 18) Leakage Stress Test: Group Split by ProductId

Stricter evaluation to reduce product-level leakage:
- Train/test split uses non-overlapping ProductId groups.
- This is a more conservative estimate than random row split.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Group-based split by ProductId (single holdout test)
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
train_idx, test_idx = next(gss.split(df["text_clean"], df["sentiment"], groups=df["ProductId"]))

X_train_g = df.iloc[train_idx]["text_clean"]
y_train_g = df.iloc[train_idx]["sentiment"]
X_test_g = df.iloc[test_idx]["text_clean"]
y_test_g = df.iloc[test_idx]["sentiment"]

print("Group split sizes:", len(X_train_g), len(X_test_g))
print("Unique ProductId train/test overlap:", len(set(df.iloc[train_idx]["ProductId"]).intersection(set(df.iloc[test_idx]["ProductId"]))))

# Train the selected winner model under group split for robustness check
group_model = best_models[winner].best_estimator_
group_model.fit(X_train_g, y_train_g)
y_pred_g = group_model.predict(X_test_g)

print("\n=== GROUP-SPLIT TEST REPORT ===")
print(classification_report(y_test_g, y_pred_g, digits=4))
print("Macro-F1 under group split is the strict robustness metric.")

Group split sizes: 484149 82960
Unique ProductId train/test overlap: 0


## 19) MongoDB Write-Back Verification (Safe)

We provide optional write-back to verify end-to-end pipeline integration with MongoDB.
Default mode is SAFE (no write). Set `ENABLE_WRITE_BACK=True` to execute.

In [ ]:
ENABLE_WRITE_BACK = False
TARGET_COLLECTION_NAME = "review_decision_dashboard"

write_payload = decision_dashboard.copy()
write_payload["generated_at"] = pd.Timestamp.now("UTC")
write_docs = write_payload.head(5000).to_dict("records")  # bounded write for safety

print(f"Prepared write-back docs: {len(write_docs):,}")
print("Target collection:", TARGET_COLLECTION_NAME)

if ENABLE_WRITE_BACK:
    target_col = client[selected_db][TARGET_COLLECTION_NAME]
    target_col.delete_many({})
    target_col.insert_many(write_docs)
    written_count = target_col.count_documents({})
    sample_doc = target_col.find_one({}, {"_id": 0})
    print("Write-back complete. Stored docs:", written_count)
    print("Sample stored document keys:", list(sample_doc.keys()) if sample_doc else [])
else:
    print("SAFE MODE: No MongoDB write performed. Set ENABLE_WRITE_BACK=True to execute.")

Prepared write-back docs: 5,000
Target collection: review_decision_dashboard
SAFE MODE: No MongoDB write performed. Set ENABLE_WRITE_BACK=True to execute.


## 20) Safe Export Helper (Avoid Permission Errors)

If files are open in another app, fallback names are used automatically.

In [ ]:
def safe_to_csv(df_obj, primary_name):
    try:
        df_obj.to_csv(primary_name, index=False)
        print(f"Saved: {primary_name}")
    except PermissionError:
        fallback = primary_name.replace(".csv", "_new.csv")
        df_obj.to_csv(fallback, index=False)
        print(f"PermissionError on {primary_name}; saved fallback: {fallback}")

safe_to_csv(val_df, "strict_model_validation_results.csv")
safe_to_csv(pd.DataFrame({"y_true": y_test.values, "y_pred": test_pred}), "strict_test_predictions.csv")
safe_to_csv(df[["Id", "ProductId", "Score", "sentiment", "text_clean"]], "strict_cleaned_full.csv")

safe_to_csv(monthly, "strict_monthly_trend.csv")
safe_to_csv(decision_table, "strict_decision_table_full.csv")
safe_to_csv(decision_dashboard, "strict_decision_dashboard.csv")
safe_to_csv(invest_top, "strict_top_invest.csv")
safe_to_csv(fix_top, "strict_top_fix.csv")
safe_to_csv(kill_top, "strict_top_kill_or_redesign.csv")

Saved: strict_model_validation_results.csv
Saved: strict_test_predictions.csv
Saved: strict_cleaned_full.csv
Saved: strict_monthly_trend.csv
Saved: strict_decision_table_full.csv
Saved: strict_decision_dashboard.csv
Saved: strict_top_invest.csv
Saved: strict_top_fix.csv
Saved: strict_top_kill_or_redesign.csv


## 21) Error Analysis Table (Misclassification Patterns)

This section analyzes model mistakes by class-pair direction (`y_true -> y_pred`).
It helps explain where the model fails most often and supports targeted improvement.

Why this matters:
- Accuracy alone can hide class-specific errors.
- Macro-F1 + error-pair analysis provides a more reliable quality view.
- We can prioritize mitigation for the most frequent error directions.

In [ ]:
# Error Analysis Table: top misclassification pairs
import pandas as pd

# Build evaluation frame from test output
err_df = pd.DataFrame({
    "y_true": y_test.values,
    "y_pred": test_pred
})

# Keep only errors
err_only = err_df[err_df["y_true"] != err_df["y_pred"]].copy()

# Pair-level aggregation: true -> predicted
pair_table = (
    err_only
    .groupby(["y_true", "y_pred"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

# Add percentages
total_test = len(err_df)
total_errors = len(err_only)
pair_table["pct_of_errors"] = (pair_table["count"] / total_errors * 100).round(2)
pair_table["pct_of_test"] = (pair_table["count"] / total_test * 100).round(2)

print(f"Total test samples: {total_test:,}")
print(f"Total misclassified: {total_errors:,}")
print(f"Error rate: {total_errors/total_test:.2%}")

print("\nTop misclassification pairs (y_true -> y_pred):")
display(pair_table.head(10))

Total test samples: 85,067
Total misclassified: 9,231
Error rate: 10.85%

Top misclassification pairs (y_true -> y_pred):


,y_true,y_pred,count,pct_of_errors,pct_of_test
5,positive,neutral,2603,28.20,3.06
4,positive,negative,1660,17.98,1.95
3,neutral,positive,1442,15.62,1.70
0,negative,neutral,1298,14.06,1.53
2,neutral,negative,1281,13.88,1.51
1,negative,positive,947,10.26,1.11
